# Chapter 8: Knowledge Distillation — Making Powerful Models Practical

This chapter covers:
- Distilling knowledge from large models
- Using temperature-scaled soft targets
- Building DeepSeek-R1's distilled models

In this chapter, we take everything our massive DeepSeek-R1 model has learned and compress it into models small enough to run on a single consumer GPU or even a phone. The key technique is **knowledge distillation** — training a small "student" model to reproduce the behavior of a larger "teacher" model.

## Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

---

## The Teacher-Student Paradigm

Instead of training a small model from scratch on raw data, we train it to **mimic a larger, more capable model**. The large model becomes the **teacher**, and the small model becomes the **student**.

### Why Soft Labels Carry More Information Than Hard Labels

A **hard label** is a one-hot vector: `[1, 0, 0, 0]` — approximately 1 bit of information per sample.

A **soft label** from the teacher: `[cat: 86%, dog: 12%, fox: 1.5%, bird: 0.5%]` — communicates a rich web of inter-class relationships. This is what Geoffrey Hinton called **"dark knowledge"** in his seminal 2015 paper.

---

## Temperature and Dark Knowledge

Under a standard softmax (T=1), the teacher's output is too peaked — dark knowledge is hidden in near-zero probabilities. **Temperature scaling** solves this by dividing logits by T before softmax:

$$q_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- **T=1**: Standard softmax (too peaked)
- **T=2–5**: The Goldilocks zone (dark knowledge emerges)
- **T≥10**: Too soft (signal lost in near-uniformity)

### Listing 8.1: Visualizing Temperature-Scaled Softmax

In [ ]:
def visualize_temperature(logits, temps):
    """Show how temperature affects softmax."""
    for T in temps:
        probs = F.softmax(logits / T, dim=-1)
        print(f"T={T:2d}: {[f'{p:.3f}' for p in probs.tolist()]}")

logits = torch.tensor([5.0, 3.0, 1.0, -1.0])
visualize_temperature(logits, [1, 2, 3, 5, 10])

Running this code shows the full progression from peaked to uniform:

```
T= 1: [0.865, 0.117, 0.016, 0.002]
T= 2: [0.657, 0.242, 0.079, 0.022]
T= 3: [0.523, 0.268, 0.138, 0.071]
T= 5: [0.421, 0.286, 0.183, 0.110]
T=10: [0.329, 0.270, 0.221, 0.181]
```

At **T=3**, the dark knowledge is clearly visible:
- "Dog" (26.8%) — the teacher sees dogs as somewhat cat-like
- "Fox" (13.8%) — foxes share some visual features with cats
- "Bird" (7.1%) — birds are the least similar to cats

### Why T² Scaling Matters

When we divide logits by T, the resulting gradients are scaled down by $1/T^2$. To compensate, we multiply the soft loss by $T^2$:

$$\mathcal{L}_{\text{soft}} = T^2 \cdot \text{KL}\big(\text{softmax}(z_t / T) \,\|\, \text{softmax}(z_s / T)\big)$$

---

## Building the Distillation Loss: From Naive to Powerful

### The Three Attempts

1. **Attempt #1**: Train from scratch with hard labels — ~1 bit of info per sample, slow learning
2. **Attempt #2**: Match teacher's output at T=1 — still too peaked, marginal improvement
3. **Attempt #3**: Temperature-scaled soft targets + hard labels — **the complete solution**

### The Complete Distillation Loss

The final loss has two components:

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{hard}} + (1 - \alpha) \cdot T^2 \cdot \text{KL}(\text{teacher\_soft} \,\|\, \text{student\_soft})$$

- **$\mathcal{L}_{\text{hard}}$**: Standard cross-entropy with ground-truth labels (grounding)
- **$\mathcal{L}_{\text{soft}}$**: KL divergence between teacher and student soft targets (knowledge transfer)
- **$\alpha$**: Controls the balance (typically 0.3 for hard, 0.7 for soft)
- **$T^2$**: Compensates for reduced gradient magnitude at higher temperatures

### Listing 8.2: The Knowledge Distillation Loss Function

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels,
                      temperature=3.0, alpha=0.3):
    """Complete knowledge distillation loss combining hard and soft targets."""
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    kl_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean')

    hard_loss = F.cross_entropy(student_logits, labels)

    return alpha * hard_loss + (1 - alpha) * temperature**2 * kl_loss

Let's verify the loss function with a quick test: